# PyTorch (.pth) to ONNX Conversion

This notebook demonstrates how to convert a trained PyTorch model to ONNX format for deployment in Flutter mobile apps using ONNX Runtime.

## Prerequisites
- Trained PyTorch model (.pth file)
- PyTorch installed
- ONNX package installed

## Step 1: Install Required Packages

In [ ]:
# Install required packages
!pip install torch torchvision onnx onnxruntime

## Step 2: Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import onnx
import onnxruntime as ort
import numpy as np
from PIL import Image
import os

print(f"PyTorch version: {torch.__version__}")
print(f"ONNX version: {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

## Step 3: Define Your Model Architecture

This should match the exact architecture you used during training. For the mushroom classifier, we use a hierarchical EfficientNet with dual classification heads.

In [ ]:
class HierarchicalMushroomClassifier(nn.Module):
    """
    Hierarchical classifier with:
    - Shared backbone (EfficientNet-B3)
    - Main class head (4 classes)
    - Species head (215 classes)
    """
    
    def __init__(self, num_main_classes=4, num_species=215, pretrained=False):
        super(HierarchicalMushroomClassifier, self).__init__()
        
        # Load EfficientNet-B3 backbone
        self.backbone = models.efficientnet_b3(pretrained=pretrained)
        
        # Get the number of features from backbone
        num_features = self.backbone.classifier[1].in_features
        
        # Remove the original classifier
        self.backbone.classifier = nn.Identity()
        
        # Main class classification head (4 classes)
        self.main_class_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_main_classes)
        )
        
        # Species classification head (215 classes)
        self.species_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(num_features, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_species)
        )
    
    def forward(self, x):
        # Extract features using backbone
        features = self.backbone(x)
        
        # Get predictions from both heads
        main_class_logits = self.main_class_head(features)
        species_logits = self.species_head(features)
        
        # Concatenate outputs: [main_class (4), species (215)] = 219 total
        return torch.cat([main_class_logits, species_logits], dim=1)

print("Model architecture defined successfully!")

## Step 4: Load the Trained Model Weights

In [ ]:
# Path to your trained model
MODEL_PATH = "assets/model/hierarchical_effnet_b3.pt"

# Number of classes
NUM_MAIN_CLASSES = 4
NUM_SPECIES = 215

# Create model instance
model = HierarchicalMushroomClassifier(
    num_main_classes=NUM_MAIN_CLASSES,
    num_species=NUM_SPECIES,
    pretrained=False
)

# Load trained weights
if os.path.exists(MODEL_PATH):
    state_dict = torch.load(MODEL_PATH, map_location='cpu')
    model.load_state_dict(state_dict)
    print(f"✓ Loaded weights from: {MODEL_PATH}")
else:
    print(f"✗ Model file not found: {MODEL_PATH}")
    print("  Please update MODEL_PATH to point to your .pth file")

# Set model to evaluation mode (important!)
model.eval()
print("✓ Model set to evaluation mode")

## Step 5: Create Dummy Input for Export

ONNX export requires a sample input to trace the model's computation graph.

In [ ]:
# Input specifications
BATCH_SIZE = 1
CHANNELS = 3       # RGB
INPUT_HEIGHT = 300
INPUT_WIDTH = 300

# Create dummy input tensor
dummy_input = torch.randn(BATCH_SIZE, CHANNELS, INPUT_HEIGHT, INPUT_WIDTH)

print(f"Dummy input shape: {dummy_input.shape}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Channels: {CHANNELS}")
print(f"  Height: {INPUT_HEIGHT}")
print(f"  Width: {INPUT_WIDTH}")

## Step 6: Test Model with Dummy Input

In [ ]:
# Test forward pass
with torch.no_grad():
    output = model(dummy_input)

print(f"Output shape: {output.shape}")
print(f"  Expected: [1, {NUM_MAIN_CLASSES + NUM_SPECIES}] = [1, 219]")
print(f"")
print(f"Output breakdown:")
print(f"  Main class logits (0:4): {output[0, :NUM_MAIN_CLASSES]}")
print(f"  Species logits (4:219): shape {output[0, NUM_MAIN_CLASSES:].shape}")

## Step 7: Export to ONNX Format

In [ ]:
# ONNX output path
ONNX_PATH = "assets/model/mushroom_model.onnx"

# Export the model
torch.onnx.export(
    model,                          # Model to export
    dummy_input,                    # Sample input
    ONNX_PATH,                      # Output file path
    export_params=True,             # Include trained weights
    opset_version=12,               # ONNX opset version
    do_constant_folding=True,       # Optimize constant expressions
    input_names=['input'],          # Input tensor name
    output_names=['output'],        # Output tensor name
    dynamic_axes={                  # Allow dynamic batch size
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"✓ Model exported to: {ONNX_PATH}")

# Check file size
file_size = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f"  File size: {file_size:.2f} MB")

## Step 8: Validate the ONNX Model

In [ ]:
# Load and check the ONNX model
onnx_model = onnx.load(ONNX_PATH)

# Verify model structure
try:
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model is valid!")
except onnx.checker.ValidationError as e:
    print(f"✗ ONNX model validation failed: {e}")

# Print model info
print(f"\nModel info:")
print(f"  IR version: {onnx_model.ir_version}")
print(f"  Opset version: {onnx_model.opset_import[0].version}")
print(f"  Producer: {onnx_model.producer_name}")

## Step 9: Print Input/Output Information

In [ ]:
# Print input details
print("Model Inputs:")
for input_tensor in onnx_model.graph.input:
    print(f"  Name: {input_tensor.name}")
    shape = [dim.dim_value if dim.dim_value > 0 else 'dynamic' 
             for dim in input_tensor.type.tensor_type.shape.dim]
    print(f"  Shape: {shape}")
    print(f"  Type: {input_tensor.type.tensor_type.elem_type}")

print("\nModel Outputs:")
for output_tensor in onnx_model.graph.output:
    print(f"  Name: {output_tensor.name}")
    shape = [dim.dim_value if dim.dim_value > 0 else 'dynamic' 
             for dim in output_tensor.type.tensor_type.shape.dim]
    print(f"  Shape: {shape}")
    print(f"  Type: {output_tensor.type.tensor_type.elem_type}")

## Step 10: Test ONNX Runtime Inference

In [ ]:
# Create ONNX Runtime session
ort_session = ort.InferenceSession(
    ONNX_PATH,
    providers=['CPUExecutionProvider']
)

# Prepare input
input_name = ort_session.get_inputs()[0].name
ort_input = dummy_input.numpy()

# Run inference
ort_output = ort_session.run(None, {input_name: ort_input})

print(f"ONNX Runtime output shape: {ort_output[0].shape}")
print(f"✓ ONNX Runtime inference successful!")

## Step 11: Compare PyTorch vs ONNX Outputs

In [ ]:
# Get PyTorch output
with torch.no_grad():
    pytorch_output = model(dummy_input).numpy()

# Get ONNX output
onnx_output = ort_output[0]

# Compare outputs
max_diff = np.max(np.abs(pytorch_output - onnx_output))
mean_diff = np.mean(np.abs(pytorch_output - onnx_output))

print(f"Comparison Results:")
print(f"  Max absolute difference: {max_diff:.8f}")
print(f"  Mean absolute difference: {mean_diff:.8f}")

if max_diff < 1e-5:
    print(f"✓ Outputs match within tolerance (1e-5)")
else:
    print(f"⚠ Outputs differ by more than 1e-5")
    print(f"  This may be due to floating-point precision differences")

## Step 12: Generate Labels File

In [ ]:
# Main class labels
MAIN_CLASSES = [
    "Non_Poisnous_Edible",
    "Non_Poisnous_Non_Edible",
    "Poisnous_Non_Useable",
    "Poisnous_Useable"
]

# Species labels (example - replace with your actual species list)
# You can load this from your training data or a CSV file
SPECIES_LABELS = [
    "almond_mushroom",
    "amethyst_chanterelle",
    "aniseed_funnel",
    # ... add all 215 species
]

# Save labels file
LABELS_PATH = "assets/model/mushroom_labels.txt"

with open(LABELS_PATH, 'w') as f:
    # Write main classes
    f.write("# Main Classes\n")
    for label in MAIN_CLASSES:
        f.write(f"{label}\n")
    
    # Write separator
    f.write("# Species\n")
    
    # Write species labels
    for label in SPECIES_LABELS:
        f.write(f"{label}\n")

print(f"✓ Labels saved to: {LABELS_PATH}")

## Step 13: Final Summary

In [ ]:
print("="*50)
print("ONNX CONVERSION SUMMARY")
print("="*50)
print(f"")
print(f"Input Model:  {MODEL_PATH}")
print(f"Output Model: {ONNX_PATH}")
print(f"Labels File:  {LABELS_PATH}")
print(f"")
print(f"Model Architecture:")
print(f"  Backbone: EfficientNet-B3")
print(f"  Main Classes: {NUM_MAIN_CLASSES}")
print(f"  Species: {NUM_SPECIES}")
print(f"  Total Output: {NUM_MAIN_CLASSES + NUM_SPECIES}")
print(f"")
print(f"Input Tensor:")
print(f"  Shape: [batch, 3, 300, 300]")
print(f"  Type: float32")
print(f"  Normalization: ImageNet (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])")
print(f"")
print(f"Output Tensor:")
print(f"  Shape: [batch, 219]")
print(f"  [0:4] = Main class logits")
print(f"  [4:219] = Species logits")
print(f"")
print(f"Files to copy to Flutter:")
print(f"  1. {ONNX_PATH}")
print(f"  2. {ONNX_PATH}.data (if generated)")
print(f"  3. {LABELS_PATH}")
print("="*50)

## Optional: Quantize Model for Smaller Size

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Quantize to INT8 for smaller model size
QUANTIZED_PATH = "assets/model/mushroom_model_quantized.onnx"

quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=QUANTIZED_PATH,
    weight_type=QuantType.QInt8
)

# Compare sizes
original_size = os.path.getsize(ONNX_PATH) / (1024 * 1024)
quantized_size = os.path.getsize(QUANTIZED_PATH) / (1024 * 1024)

print(f"Original model size: {original_size:.2f} MB")
print(f"Quantized model size: {quantized_size:.2f} MB")
print(f"Size reduction: {(1 - quantized_size/original_size)*100:.1f}%")